# Geocoding with Google

In [3]:
import os
import time
import requests
import pandas as pd
from pandas.errors import EmptyDataError, ParserError

In [ ]:
API_KEY = "x"
EXCEL_PATH = r"C:\Users\lucyq\Dropbox\AMDP\THESIS\AddressBulk2.xlsx"
OUTPUT_CSV = r"C:\Users\lucyq\Dropbox\AMDP\THESIS\AddressBulk2_geocoded.csv"

# --- Load addresses (single column named 'address') ---
df = pd.read_excel(EXCEL_PATH, dtype=str, usecols=["address"])
df["address"] = df["address"].fillna("").str.strip()
df = df[df["address"] != ""]  # drop blanks

# unique list to save quota
unique_addresses = df["address"].drop_duplicates().tolist()

# Optional: bias (but not force) results to Rome area; strict filter to Italy
BOUNDS_SW = "41.6,12.1"  # lat,lng
BOUNDS_NE = "42.1,12.8"  # lat,lng


def geocode_one(addr: str) -> dict:
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        "address": addr,
        "key": API_KEY,
        "components": "country:IT",                 # strict to Italy
        "language": "it",                           # response language
        "region": "it",                             # country bias
        "bounds": f"{BOUNDS_SW}|{BOUNDS_NE}",       # viewport bias around Rome
    }
    backoff = 0.5
    for _ in range(6):
        r = requests.get(url, params=params, timeout=30)
        if r.status_code == 429:
            # throttle/backoff if needed
            time.sleep(backoff)
            backoff = min(backoff * 2, 30)
            continue
        r.raise_for_status()
        data = r.json()
        status = data.get("status", "")
        results = data.get("results", [])
        if status == "OK" and results:
            top = results[0]
            loc = top["geometry"]["location"]
            return {
                "lat": loc.get("lat"),
                "lng": loc.get("lng"),
                "formatted_address": top.get("formatted_address"),
                "place_id": top.get("place_id"),
                "types": ",".join(top.get("types", [])),
                "status": status,
            }
        elif status in {"ZERO_RESULTS", "OVER_QUERY_LIMIT", "REQUEST_DENIED", "INVALID_REQUEST"}:
            return {"lat": None, "lng": None, "formatted_address": None,
                    "place_id": None, "types": None, "status": status}
        # transient/UNKNOWN_ERROR: retry with backoff
        time.sleep(backoff)
        backoff = min(backoff * 2, 30)
    return {"lat": None, "lng": None, "formatted_address": None,
            "place_id": None, "types": None, "status": "RETRIES_EXHAUSTED"}


# --- Geocode uniques and build a map (address -> fields) ---
result_map = {}
for i, a in enumerate(unique_addresses, start=1):
    out = geocode_one(a)
    result_map[a] = out
    if i % 50 == 0:
        print(f"Geocoded {i}/{len(unique_addresses)} uniques...")
    time.sleep(0.25)  # ~4 req/s to be gentle

# --- Map results back to *every* row (no merge needed) ---
df["lat"] = df["address"].map(lambda x: result_map.get(x, {}).get("lat"))
df["lng"] = df["address"].map(lambda x: result_map.get(x, {}).get("lng"))
df["formatted_address"] = df["address"].map(
    lambda x: result_map.get(x, {}).get("formatted_address"))
df["place_id"] = df["address"].map(
    lambda x: result_map.get(x, {}).get("place_id"))
df["types"] = df["address"].map(lambda x: result_map.get(x, {}).get("types"))
df["status"] = df["address"].map(lambda x: result_map.get(x, {}).get("status"))

df.to_csv(OUTPUT_CSV, index=False)
print(f"Done. Wrote {len(df)} rows to {OUTPUT_CSV}")
print(df.head())

Geocoded 50/743 uniques...
Geocoded 100/743 uniques...
Geocoded 150/743 uniques...
Geocoded 200/743 uniques...
Geocoded 250/743 uniques...
Geocoded 300/743 uniques...
Geocoded 350/743 uniques...
Geocoded 400/743 uniques...
Geocoded 450/743 uniques...
Geocoded 500/743 uniques...
Geocoded 550/743 uniques...
Geocoded 600/743 uniques...
Geocoded 650/743 uniques...
Geocoded 700/743 uniques...
Done. Wrote 853 rows to C:\Users\lucyq\Dropbox\AMDP\THESIS\AddressBulk2_geocoded.csv
                                     address        lat        lng  \
0       VIA FRANCO SACCHETTI 1, Roma, Italia  41.951504  12.547581   
1        VIA ALFONSO BORELLI 3, Roma, Italia  41.907398  12.515550   
2  VIA DI PORTA CAVALLEGGERI 3, Roma, Italia  41.900100  12.456136   
3              VIA CONCORDIA 4, Roma, Italia  41.877747  12.508633   
4           VIA POGGIO VERDE 4, Roma, Italia  41.927225  13.091224   

                                   formatted_address  \
0     Via Franco Sacchetti, 1, 00137 Roma RM, I

In [ ]:
metadata = {
    'notebook'                : '010ii Geocoding via Google.ipynb',
    'step'                    : 'Google Maps geocoding + validation + merge-back',

    # IO lineage
    'input_address_source'    : '<fill: e.g., addresses_to_geocode.parquet>',
    'merge_target_file'       : '<fill: e.g., 009_weather_included.csv>',
    'output_parquet_file'     : '<fill: e.g., 010ii_gmaps_geocoded.parquet>',
    'output_excel_file'       : '<optional>',
    'output_csv_file'         : '<optional>',

    # Shapes
    'input_rows'              : '<fill>',
    'input_columns'           : '<fill>',
    'rows_attempted_geocode'  : '<fill>',
    'rows_already_cached'     : '<fill>',
    'rows_api_called'         : '<fill>',
    'rows_success'            : '<fill>',
    'rows_failed'             : '<fill>',
    'final_rows'              : '<fill>',
    'final_columns'           : '<fill>',

    # Address normalization (before API)
    'normalized_full_address_col': 'address_norm',
    'normalization_rules'     : [
        'strip/uppercase/collapse spaces',
        'expand Via/V.le/Viale; P.zza/Piazza',
        'remove km markers/trailing commas',
        'force Comune="Roma", Provincia="RM", Nazione="Italia" when implied'
    ],

    # Google Geocoding API configuration
    'geocode_provider'        : 'Google Maps Geocoding API',
    'api_key_env'             : 'GOOGLE_MAPS_API_KEY',
    'language'                : 'it',
    'region'                  : 'IT',
    'components_filter'       : 'country:IT|locality:Roma',
    'location_bias'           : {'type':'circle', 'center':[41.9028,12.4964], 'radius_km':25},
    'rate_limit_qps'          : '<fill: e.g., 5>',
    'retry_policy'            : {'strategy':'exponential_backoff_with_jitter', 'max_retries':3},
    'timeout_seconds'         : 10,

    # Caching
    'cache_used'              : True,
    'cache_file'              : '<fill: e.g., gmaps_cache.parquet>',
    'cache_key'               : 'address_norm',
    'cache_hit_rate_%'        : '<fill>',

    # Columns added (from Google)
    'columns_added'           : [
        'Latitude_gmaps','Longitude_gmaps',
        'gmaps_precision',              # ROOFTOP, RANGE_INTERPOLATED, GEOMETRIC_CENTER, APPROXIMATE
        'gmaps_place_id',
        'gmaps_formatted_address',
        'gmaps_types',                  # list/primary type
        'gmaps_partial_match',          # bool
        'gmaps_viewport',               # NE/SW bbox
        'gmaps_geocode_status',         # OK/ZERO_RESULTS/...
        'gmaps_response_ts',            # timestamp
        'address_norm'
    ],
    'columns_removed'         : [],

    # Merge-back policy (into master dataset)
    'merge_keys'              : ['Protocollo'],  # adjust if you merge on another key
    'crs'                     : 'WGS84 (EPSG:4326)',
    'coord_precision_digits'  : 6,
    'precision_rank_order'    : ['ROOFTOP','RANGE_INTERPOLATED','GEOMETRIC_CENTER','APPROXIMATE'],
    'merge_policy'            : (
        "Prefer Google coords if gmaps_precision outranks existing source; "
        "keep previous coords otherwise. Never override with partial_match=True unless manually reviewed."
    ),

    # QA & validation
    'qa_checks'               : {
        'invalid_coords_after'      : 0,
        'outside_rome_bbox_count'   : '<fill>',
        'distance_vs_prev_m'        : {'median':'<fill>', 'p95':'<fill>', 'max':'<fill>'},
        'duplicate_place_ids'       : '<fill>',
        'ambiguous_candidates_count': '<fill>',
        'partial_match_count'       : '<fill>',
        'status_counts'             : {
            'OK':'<fill>', 'ZERO_RESULTS':'<fill>',
            'OVER_QUERY_LIMIT':'<fill>', 'REQUEST_DENIED':'<fill>',
            'INVALID_REQUEST':'<fill>', 'UNKNOWN_ERROR':'<fill>'
        }
    },

    # Cost & quota (fill if you tracked)
    'api_calls_made'          : '<fill>',
    'unit_price_eur'          : '<optional>',
    'cost_estimate_eur'       : '<optional>',
    'billing_notes'           : 'Pay-as-you-go; cache to minimize repeat calls',

    # Decisions & notes
    'decisions_made'          : [
        'Standardised addresses to maximise cache hits and stable geocoding.',
        'Restricted search to Rome/IT via components and location bias.',
        'Ranked candidates by precision; rejected coarse results and suspicious partial matches.',
        'Preserved previous coordinates when Google result was less precise or out-of-bounds.',
        'Stored place_i_
